# Chapter 15 — Processing Sequences Using RNNs and CNNs
Companion notebook to `part2_deep_learning/15_rnn_sequences_time_series.md`. No downloads needed
(synthetic time series).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

## Generate a synthetic time series

In [ ]:
def generate_time_series(batch_size, n_steps):
    freq1, freq2, offsets1, offsets2 = np.random.rand(4, batch_size, 1)
    time = np.linspace(0, 1, n_steps)
    series = 0.5 * np.sin((time - offsets1) * (freq1 * 10 + 10))
    series += 0.2 * np.sin((time - offsets2) * (freq2 * 20 + 20))
    series += 0.1 * (np.random.rand(batch_size, n_steps) - 0.5)
    return series[..., np.newaxis].astype(np.float32)

n_steps = 50
series = generate_time_series(10000, n_steps + 1)
X_train, y_train = series[:7000, :n_steps], series[:7000, -1]
X_valid, y_valid = series[7000:9000, :n_steps], series[7000:9000, -1]
X_test, y_test = series[9000:, :n_steps], series[9000:, -1]

plt.plot(series[0])
plt.title("One example time series")
plt.show()

## Baseline: naive forecast & a plain linear model

In [ ]:
y_pred_naive = X_valid[:, -1]
naive_mse = np.mean((y_pred_naive - y_valid) ** 2)
print("Naive forecast MSE:", naive_mse)

linear_model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[n_steps, 1]),
    keras.layers.Dense(1),
])
linear_model.compile(loss="mse", optimizer="adam")
linear_model.fit(X_train, y_train, epochs=10, validation_data=(X_valid, y_valid), verbose=0)
print("Linear model MSE:", linear_model.evaluate(X_valid, y_valid))

## SimpleRNN vs LSTM vs GRU

In [ ]:
def build_and_eval(cell_type):
    Cell = {"SimpleRNN": keras.layers.SimpleRNN, "LSTM": keras.layers.LSTM, "GRU": keras.layers.GRU}[cell_type]
    model = keras.models.Sequential([
        Cell(20, return_sequences=True, input_shape=[None, 1]),
        Cell(20),
        keras.layers.Dense(1),
    ])
    model.compile(loss="mse", optimizer="adam")
    model.fit(X_train, y_train, epochs=10, validation_data=(X_valid, y_valid), verbose=0)
    mse = model.evaluate(X_valid, y_valid, verbose=0)
    print(f"{cell_type:10s} validation MSE = {mse:.5f}")
    return model

for cell_type in ["SimpleRNN", "LSTM", "GRU"]:
    build_and_eval(cell_type)

## ✍️ Practice
1. Compare naive baseline, linear, SimpleRNN, LSTM, GRU MSE above — which wins, by how much?
2. In your own words, explain what each of the LSTM's three gates is "deciding."
3. Try forecasting 10 steps ahead instead of 1 (either by feeding predictions back in, or training a multi-output model).
4. Debug: SimpleRNN performs well on 10-step sequences but badly on 200-step sequences even with more units — what's the likely cause, and the fix?

In [ ]:
# Your code / notes here